# 4 Modelling Open Screening

## 4.0 Setup

In [1]:
# basic imports
from pathlib import Path
import json
import time
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [2]:
# sklearn imports
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    balanced_accuracy_score,
    brier_score_loss,
    log_loss
)

In [3]:
# project paths
PROJECT_ROOT = Path.home() / "Documents" / "Thesis"
DATA_DIR = PROJECT_ROOT / "Data"
PROCESSED_DATA_DIR = DATA_DIR / "Processed Data"
OUTPUT_DIR = PROJECT_ROOT / "Outputs"

OPEN_MODEL_PATH = PROCESSED_DATA_DIR / "df_open_model_v1.parquet"

In [4]:
# modelling constants
TARGET_COL = "open"
GROUP_COL = "mailing_id"

FINAL_TEST_SIZE = 0.20
INNER_CV_SPLITS = 5

In [5]:
# safety check
print("Project root:", PROJECT_ROOT)
print("Processed data folder exists:", PROCESSED_DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Open model file exists:", OPEN_MODEL_PATH.exists())

Project root: /Users/khanhhien/Documents/Thesis
Processed data folder exists: True
Output folder exists: True
Open model file exists: True


## 4.1 Load open dataset

In [6]:
# load open dataset
df_open = pd.read_parquet(OPEN_MODEL_PATH)

print("df_open shape:", df_open.shape)
df_open.head()

df_open shape: (1057544, 38)


,user_id,mailing_id,open,click,mailing_info,subject_line,preheader,send_date,send_day_of_week,date_trusted,...,preheader_contains_exclamation,prior_email_count,prior_open_count,prior_click_count,historical_open_rate,historical_click_rate,historical_open_bucket,prior_email_frequency_bucket,had_prior_click,is_first_email
0,5c6bebde5798a794283224c9,668,0.0,0.0,2025/04/24 CPX - MM MAX Magazine - 4 gratis nrs,"Lezen, puzzelen, genieten: 4 edities cadeau","Vraag nu aan – geen kosten, geen verplichtingen.",2025-04-25,Vrijdag,True,...,False,0,0.0,0.0,NaN,NaN,NaN,very_little,False,True
1,5c6bebde5798a794283224c9,691,0.0,0.0,2025/05/29 WOONCOMFORT AIRCO,Bespaar tot 80% op je energiekosten met een airco,Vraag nu gratis maatwerkadvies aan voor energi...,2025-04-30,Woensdag,True,...,False,1,0.0,0.0,0.0,0.0,very_low,very_little,False,False
2,5c6bebde5798a794283224c9,714,0.0,0.0,2025/05/06 CPX - MM - ENGIE BONUS,Profiteer nu: €300 bonus én vaste energietarieven,ENGIE helpt je graag met persoonlijk advies.,2025-05-07,Woensdag,True,...,False,2,0.0,0.0,0.0,0.0,very_low,very_little,False,False
3,5c6bebde5798a794283224c9,733,0.0,0.0,2025/05/14 LIFEPOINTS,Jouw mening is geld waard – start vandaag nog,Verdien punten voor digitale cadeaubonnen zoal...,2025-05-14,Woensdag,True,...,False,3,0.0,0.0,0.0,0.0,very_low,very_little,False,False
4,5c6bebde5798a794283224c9,761,0.0,0.0,2025/05/27 CPX - MM - ENGIE,Speel mee met de ENGIE woordlegger!,"En win een solar powerbank t.w.v. €79,95.",2025-06-04,Woensdag,True,...,False,4,0.0,0.0,0.0,0.0,very_low,very_little,False,False


In [8]:
# basic target checks
print("Target column:", TARGET_COL)
print("Missing target values:", df_open[TARGET_COL].isna().sum())

display(df_open[TARGET_COL].value_counts(dropna=False))
display(df_open[TARGET_COL].value_counts(normalize=True, dropna=False))

Target column: open
Missing target values: 0


open
1.0    567034
0.0    490510
Name: count, dtype: int64

open
1.0    0.53618
0.0    0.46382
Name: proportion, dtype: float64

In [9]:
# campaign checks
print("Number of unique campaigns:", df_open[GROUP_COL].nunique())
print("Campaign ID range:", df_open[GROUP_COL].min(), "to", df_open[GROUP_COL].max())

display(df_open[GROUP_COL].describe())

Number of unique campaigns: 274
Campaign ID range: 653 to 1418


count      1057544.0
mean     1053.667041
std       222.845854
min            653.0
25%            874.0
50%           1031.0
75%           1274.0
max           1418.0
Name: mailing_id, dtype: Float64

In [10]:
# column overview
print("Number of columns:", df_open.shape[1])
display(pd.DataFrame({
    "column": df_open.columns,
    "dtype": [df_open[col].dtype for col in df_open.columns],
    "missing_rate": [df_open[col].isna().mean() for col in df_open.columns]
}).sort_values("missing_rate", ascending=False))

Number of columns: 38


,column,dtype,missing_rate
13,age_group,category,0.745815
12,age_clean,float64,0.745815
8,send_day_of_week,object,0.329770
7,send_date,datetime64[us],0.329770
33,historical_click_rate,float64,0.060171
3,click,float64,0.043825
31,prior_click_count,float64,0.043825
6,preheader,object,0.029937
5,subject_line,object,0.025269
4,mailing_info,object,0.025269


In [11]:
# inspect columns

# column list
for i, col in enumerate(df_open.columns):
    print(f"{i:2d}: {col}")

# verify critical columns

critical_cols = [
    "mailing_id",
    "open",
    "campaign_topic",
    "gender",
    "age_group",
    "historical_open_bucket",
    "prior_email_frequency_bucket"
]

for col in critical_cols:
    print(f"{col}: {col in df_open.columns}")

 0: user_id
 1: mailing_id
 2: open
 3: click
 4: mailing_info
 5: subject_line
 6: preheader
 7: send_date
 8: send_day_of_week
 9: date_trusted
10: has_campaign_metadata
11: gender
12: age_clean
13: age_group
14: n_interests
15: interest_topic_match
16: campaign_topic
17: subject_missing
18: subject_length
19: subject_length_group
20: subject_contains_number
21: subject_contains_promotion
22: subject_contains_exclamation
23: preheader_missing
24: preheader_length
25: preheader_length_group
26: preheader_contains_number
27: preheader_contains_promotion
28: preheader_contains_exclamation
29: prior_email_count
30: prior_open_count
31: prior_click_count
32: historical_open_rate
33: historical_click_rate
34: historical_open_bucket
35: prior_email_frequency_bucket
36: had_prior_click
37: is_first_email
mailing_id: True
open: True
campaign_topic: True
gender: True
age_group: True
historical_open_bucket: True
prior_email_frequency_bucket: True


## 4.2 Recreate open feature sets and preprocessing design

In [12]:
# define feature type dictionaries

feature_types_A = {
    "numeric": [
        "n_interests"
    ],

    "categorical": [
        "gender",
        "age_group",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_bucket",
        "prior_email_frequency_bucket"
    ],

    "binary": [
        "interest_topic_match",
        "has_campaign_metadata",
        "subject_contains_number",
        "subject_contains_promotion",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "is_first_email"
    ]
}

feature_types_B = {
    "numeric": feature_types_A["numeric"].copy(),

    "categorical": (
        feature_types_A["categorical"].copy()
        + [
            "campaign_topic_x_age_group",
            "historical_open_bucket_x_prior_email_frequency_bucket"
        ]
    ),

    "binary": feature_types_A["binary"].copy()
}

feature_types_C = {
    "numeric": [
        "age_clean",
        "n_interests",
        "subject_length",
        "preheader_length",
        "prior_email_count",
        "prior_open_count",
        "prior_click_count",
        "historical_open_rate",
        "historical_click_rate"
    ],

    "categorical": [
        "gender",
        "age_group",
        "campaign_topic",
        "subject_length_group",
        "preheader_length_group",
        "historical_open_bucket",
        "prior_email_frequency_bucket"
    ],

    "binary": [
        "interest_topic_match",
        "has_campaign_metadata",
        "subject_missing",
        "subject_contains_number",
        "subject_contains_promotion",
        "subject_contains_exclamation",
        "preheader_missing",
        "preheader_contains_number",
        "preheader_contains_promotion",
        "preheader_contains_exclamation",
        "had_prior_click",
        "is_first_email"
    ]
}

In [13]:
# derive feature sets from feature type dictionaries
def flatten_feature_types(feature_types):
    return (
        feature_types["numeric"]
        + feature_types["categorical"]
        + feature_types["binary"]
    )

feature_set_A_core = flatten_feature_types(feature_types_A)
feature_set_B_interactions = flatten_feature_types(feature_types_B)
feature_set_C_expanded = flatten_feature_types(feature_types_C)

feature_sets = {
    "A_core": feature_set_A_core,
    "B_interactions": feature_set_B_interactions,
    "C_expanded": feature_set_C_expanded
}

feature_type_sets = {
    "A_core": feature_types_A,
    "B_interactions": feature_types_B,
    "C_expanded": feature_types_C
}

for name, features in feature_sets.items():
    print(name, ":", len(features), "features")

A_core : 15 features
B_interactions : 17 features
C_expanded : 28 features


In [14]:
# create deterministic interaction columns
def combine_interaction_cols(df, col1, col2):
    return (
        df[col1].astype("object").fillna("Missing").astype(str)
        + " x "
        + df[col2].astype("object").fillna("Missing").astype(str)
    )

df_open["campaign_topic_x_age_group"] = combine_interaction_cols(
    df_open, "campaign_topic", "age_group"
)

df_open["historical_open_bucket_x_prior_email_frequency_bucket"] = combine_interaction_cols(
    df_open, "historical_open_bucket", "prior_email_frequency_bucket"
)

interaction_cols = [
    "campaign_topic_x_age_group",
    "historical_open_bucket_x_prior_email_frequency_bucket"
]

for col in interaction_cols:
    print(col)
    print("  exists:", col in df_open.columns)
    print("  missing rate:", df_open[col].isna().mean())
    print("  unique values:", df_open[col].nunique())

campaign_topic_x_age_group
  exists: True
  missing rate: 0.0
  unique values: 60
historical_open_bucket_x_prior_email_frequency_bucket
  exists: True
  missing rate: 0.0
  unique values: 26


In [15]:
# validate feature sets
def validate_feature_set(name, features, df):
    missing_features = [col for col in features if col not in df.columns]
    duplicated_features = pd.Series(features)[pd.Series(features).duplicated()].tolist()
    
    print("=" * 60)
    print(name)
    print("Number of features:", len(features))
    print("Missing features:", missing_features)
    print("Duplicated features:", duplicated_features)

for name, features in feature_sets.items():
    validate_feature_set(name, features, df_open)

A_core
Number of features: 15
Missing features: []
Duplicated features: []
B_interactions
Number of features: 17
Missing features: []
Duplicated features: []
C_expanded
Number of features: 28
Missing features: []
Duplicated features: []


In [16]:
# leakage check
forbidden_feature_cols = [
    "open",
    "click",
    "user_id",
    "mailing_id",
    "mailing_info",
    "subject_line",
    "preheader",
    "send_date",
    "send_day_of_week",
    "date_trusted"
]

for name, features in feature_sets.items():
    forbidden_found = [col for col in features if col in forbidden_feature_cols]
    print(name, "forbidden columns found:", forbidden_found)

A_core forbidden columns found: []
B_interactions forbidden columns found: []
C_expanded forbidden columns found: []


In [17]:
# build preprocessing recipes
def build_preprocessor(feature_types):
    numeric_features = feature_types["numeric"]
    categorical_features = feature_types["categorical"]
    binary_features = feature_types["binary"]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value=0)),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    binary_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_transformer, numeric_features),
            ("categorical", categorical_transformer, categorical_features),
            ("binary", binary_transformer, binary_features)
        ],
        remainder="drop"
    )

    return preprocessor


preprocessors = {
    name: build_preprocessor(feature_types)
    for name, feature_types in feature_type_sets.items()
}

preprocessors.keys()

dict_keys(['A_core', 'B_interactions', 'C_expanded'])

## 4.3 Recreate temporal campaign holdout

In [18]:
# define chronological campaign holdout
campaign_ids_sorted = np.sort(df_open[GROUP_COL].dropna().unique())

n_campaigns = len(campaign_ids_sorted)
n_test_campaigns = int(np.ceil(n_campaigns * FINAL_TEST_SIZE))
n_trainval_campaigns = n_campaigns - n_test_campaigns

trainval_campaigns = campaign_ids_sorted[:n_trainval_campaigns]
test_campaigns = campaign_ids_sorted[n_trainval_campaigns:]

print("Total campaigns:", n_campaigns)
print("Train-validation campaigns:", len(trainval_campaigns))
print("Final test campaigns:", len(test_campaigns))

print("Trainval campaign range:", trainval_campaigns.min(), "to", trainval_campaigns.max())
print("Test campaign range:", test_campaigns.min(), "to", test_campaigns.max())

Total campaigns: 274
Train-validation campaigns: 219
Final test campaigns: 55
Trainval campaign range: 653 to 1252
Test campaign range: 1257 to 1418


In [19]:
# create train-validation and final test dataframs
trainval_df = df_open[df_open[GROUP_COL].isin(trainval_campaigns)].copy()
test_df = df_open[df_open[GROUP_COL].isin(test_campaigns)].copy()

print("Train-validation shape:", trainval_df.shape)
print("Final test shape:", test_df.shape)

Train-validation shape: (781826, 40)
Final test shape: (275718, 40)


In [20]:
# check campaign overlap
trainval_campaign_set = set(trainval_df[GROUP_COL].unique())
test_campaign_set = set(test_df[GROUP_COL].unique())

campaign_overlap = trainval_campaign_set.intersection(test_campaign_set)

print("Campaign overlap:", len(campaign_overlap))
print("Overlap values:", sorted(campaign_overlap)[:10])

assert len(campaign_overlap) == 0, "Leakage: campaign appears in both trainval and test."

Campaign overlap: 0
Overlap values: []


In [21]:
# check target distribution by split
split_summary = pd.DataFrame({
    "split": ["trainval", "test"],
    "n_rows": [len(trainval_df), len(test_df)],
    "n_campaigns": [
        trainval_df[GROUP_COL].nunique(),
        test_df[GROUP_COL].nunique()
    ],
    "open_rate": [
        trainval_df[TARGET_COL].mean(),
        test_df[TARGET_COL].mean()
    ],
    "open_positive_count": [
        int(trainval_df[TARGET_COL].sum()),
        int(test_df[TARGET_COL].sum())
    ],
    "open_negative_count": [
        int((trainval_df[TARGET_COL] == 0).sum()),
        int((test_df[TARGET_COL] == 0).sum())
    ]
})

display(split_summary)

,split,n_rows,n_campaigns,open_rate,open_positive_count,open_negative_count
0,trainval,781826,219,0.573273,448200,333626
1,test,275718,55,0.430998,118834,156884


In [22]:
# store split summary
split_summary_open = split_summary.copy()
split_summary_open

,split,n_rows,n_campaigns,open_rate,open_positive_count,open_negative_count
0,trainval,781826,219,0.573273,448200,333626
1,test,275718,55,0.430998,118834,156884


## 4.4 Recreate interaction columns

Already created in 4.2

## 4.5 Set up grouped cross-validation

In [23]:
# create reference X, y, groups for CV
X_reference = trainval_df[feature_set_A_core].copy()
y_reference = trainval_df[TARGET_COL].copy()
groups_reference = trainval_df[GROUP_COL].copy()

print("X_reference shape:", X_reference.shape)
print("y_reference shape:", y_reference.shape)
print("Number of groups:", groups_reference.nunique())
print("Open rate:", y_reference.mean())

X_reference shape: (781826, 15)
y_reference shape: (781826,)
Number of groups: 219
Open rate: 0.5732733370340716


In [24]:
# create grouped stratified CV object
cv_strategy = StratifiedGroupKFold(
    n_splits=INNER_CV_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

cv_strategy

StratifiedGroupKFold(n_splits=5, random_state=42, shuffle=True)

In [25]:
# check inner CV fold quality
fold_summaries = []

for fold_idx, (train_idx, val_idx) in enumerate(
    cv_strategy.split(X_reference, y_reference, groups_reference),
    start=1
):
    y_train_fold = y_reference.iloc[train_idx]
    y_val_fold = y_reference.iloc[val_idx]
    
    groups_train_fold = groups_reference.iloc[train_idx]
    groups_val_fold = groups_reference.iloc[val_idx]
    
    campaign_overlap = set(groups_train_fold).intersection(set(groups_val_fold))
    
    fold_summaries.append({
        "fold": fold_idx,
        "train_rows": len(train_idx),
        "val_rows": len(val_idx),
        "train_campaigns": groups_train_fold.nunique(),
        "val_campaigns": groups_val_fold.nunique(),
        "campaign_overlap": len(campaign_overlap),
        "train_open_rate": y_train_fold.mean(),
        "val_open_rate": y_val_fold.mean(),
        "val_open_positive_count": int(y_val_fold.sum()),
        "val_open_negative_count": int((y_val_fold == 0).sum())
    })

cv_fold_summary_open = pd.DataFrame(fold_summaries)

display(cv_fold_summary_open)

,fold,train_rows,val_rows,train_campaigns,val_campaigns,campaign_overlap,train_open_rate,val_open_rate,val_open_positive_count,val_open_negative_count
0,1,622701,159125,173,46,0,0.561046,0.621122,98836,60289
1,2,640613,141213,180,39,0,0.575329,0.563949,79637,61576
2,3,598719,183107,167,52,0,0.572579,0.575543,105386,77721
3,4,616035,165791,180,39,0,0.570283,0.584386,96886,68905
4,5,649236,132590,176,43,0,0.586451,0.508749,67455,65135


In [26]:
# assert fold safety

assert (cv_fold_summary_open["campaign_overlap"] == 0).all(), "Campaign leakage inside CV folds."
assert (cv_fold_summary_open["val_open_positive_count"] > 0).all(), "A validation fold has no positive open cases."
assert (cv_fold_summary_open["val_open_negative_count"] > 0).all(), "A validation fold has no negative open cases."

print("CV fold safety checks passed.")

CV fold safety checks passed.


## 4.6 Define metrics

In [27]:
# define metric calculation function
def calculate_binary_metrics(y_true, y_pred, y_proba):
    """
    Calculate binary classification metrics for one validation fold.
    
    y_true  = true labels
    y_pred  = predicted class labels
    y_proba = predicted probability for class 1
    """
    metrics = {
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        "PR_AUC": average_precision_score(y_true, y_proba),
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "brier": brier_score_loss(y_true, y_proba),
        "log_loss": log_loss(y_true, y_proba, labels=[0, 1])
    }
    
    return metrics

In [28]:
# test metric function on simple fake predictions

y_fake = pd.Series([0, 0, 1, 1])
pred_fake = pd.Series([0, 0, 1, 1])
proba_fake = pd.Series([0.1, 0.2, 0.8, 0.9])

calculate_binary_metrics(y_fake, pred_fake, proba_fake)

{'ROC_AUC': 1.0,
 'PR_AUC': 1.0,
 'accuracy': 1.0,
 'balanced_accuracy': 1.0,
 'brier': 0.024999999999999998,
 'log_loss': 0.16425203348601802}

In [29]:
# define metric column order
metric_names = [
    "ROC_AUC",
    "PR_AUC",
    "accuracy",
    "balanced_accuracy",
    "brier",
    "log_loss"
]

metric_names

['ROC_AUC', 'PR_AUC', 'accuracy', 'balanced_accuracy', 'brier', 'log_loss']

## 4.7 Run dummy baselines

In [39]:
# cross-validation evaluation helper
def evaluate_cv_model(
    model,
    X,
    y,
    groups,
    cv_strategy,
    metric_names
):
    
    fold_results = []

    for fold_idx, (train_idx, val_idx) in enumerate(
        cv_strategy.split(X, y, groups),
        start=1
    ):

        X_train = X.iloc[train_idx]
        X_val = X.iloc[val_idx]

        y_train = y.iloc[train_idx]
        y_val = y.iloc[val_idx]

        model.fit(X_train, y_train)

        y_pred = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]

        metrics = calculate_binary_metrics(
            y_true=y_val,
            y_pred=y_pred,
            y_proba=y_proba
        )

        metrics["fold"] = fold_idx

        fold_results.append(metrics)

    fold_results_df = pd.DataFrame(fold_results)

    return fold_results_df

In [40]:
X_A = trainval_df[feature_set_A_core]

y_A = trainval_df[TARGET_COL]

groups_A = trainval_df[GROUP_COL]

### 4.7.1 Majority-class dummy baseline

Predict OPEN for everyone

Probability = 1 for everyone

In [47]:
majority_dummy = DummyClassifier(
    strategy="most_frequent"
)

majority_cv_results = evaluate_cv_model(
    model=majority_dummy,
    X=X_A,
    y=y_A,
    groups=groups_A,
    cv_strategy=cv_strategy,
    metric_names=metric_names
)

display(majority_cv_results)

,ROC_AUC,PR_AUC,accuracy,balanced_accuracy,brier,log_loss,fold
0,0.5,0.621122,0.621122,0.5,0.378878,13.656156,1
1,0.5,0.563949,0.563949,0.5,0.436051,15.716853,2
2,0.5,0.575543,0.575543,0.5,0.424457,15.298972,3
3,0.5,0.584386,0.584386,0.5,0.415614,14.980234,4
4,0.5,0.508749,0.508749,0.5,0.491251,17.706489,5


In [48]:
# majority baseline summary
majority_summary = {
    metric: majority_cv_results[metric].mean()
    for metric in metric_names
}

pd.DataFrame([majority_summary])

,ROC_AUC,PR_AUC,accuracy,balanced_accuracy,brier,log_loss
0,0.5,0.57075,0.57075,0.5,0.42925,15.471741


### 4.7.2 Prior probability dummy baseline

Predict class according to training distribution

P(open) = training open rate

P(not open) = training non-open rate

In [49]:
prior_dummy = DummyClassifier(
    strategy="prior"
)

prior_cv_results = evaluate_cv_model(
    model=prior_dummy,
    X=X_A,
    y=y_A,
    groups=groups_A,
    cv_strategy=cv_strategy,
    metric_names=metric_names
)

display(prior_cv_results)

,ROC_AUC,PR_AUC,accuracy,balanced_accuracy,brier,log_loss,fold
0,0.5,0.621122,0.621122,0.5,0.238939,0.670932,1
1,0.5,0.563949,0.563949,0.5,0.246040,0.685210,2
2,0.5,0.575543,0.575543,0.5,0.244302,0.681708,3
3,0.5,0.584386,0.584386,0.5,0.243078,0.679244,4
4,0.5,0.508749,0.508749,0.5,0.255961,0.705267,5


In [50]:
prior_summary = {
    metric: prior_cv_results[metric].mean()
    for metric in metric_names
}

pd.DataFrame([prior_summary])

,ROC_AUC,PR_AUC,accuracy,balanced_accuracy,brier,log_loss
0,0.5,0.57075,0.57075,0.5,0.245664,0.684472


### 4.7.3 Summary dummy baselines

In [51]:
baseline_results_df = pd.DataFrame([
    {
        "model_name": "dummy_majority",
        **majority_summary
    },
    {
        "model_name": "dummy_prior",
        **prior_summary
    }
])

baseline_results_df

,model_name,ROC_AUC,PR_AUC,accuracy,balanced_accuracy,brier,log_loss
0,dummy_majority,0.5,0.57075,0.57075,0.5,0.429250,15.471741
1,dummy_prior,0.5,0.57075,0.57075,0.5,0.245664,0.684472


- ROC-AUC = 0.5 -> If we randomly pick one opener and one non-opener, the probability the model ranks the opener higher is random guessing (0.5) -> Dummy model cannot distinguish openers from non-openers at all.
- PR-AUC = 0.57 -> If I try to find openers, how good am I. 0.57 vs 56.7% open rate -> Dummy model has no skill beyond random selection.
- Accuracy = 0.57 -> Dummy predicts everyone opens since ~57% actually open -> Model is exploiting class imbalance.
- Balanced accuracy = 0.5 -> Random guessing -> Model actually has zero discrimination ability.
- Brier score: how far predicted probabilities from reality, lower is better -> Prior dummy gives much more reasonable probability estimates.
- Log loss: lower is better

## 4.8 Build model registry

In [52]:
# optional XGBoost import
try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    print("XGBoost available:", XGBOOST_AVAILABLE)
except ImportError:
    XGBOOST_AVAILABLE = False
    print("XGBoost available:", XGBOOST_AVAILABLE)

XGBoost available: False


In [53]:
# define model registry

model_registry = {
    "logistic_regression": LogisticRegression(
        penalty="l2",
        C=1.0,
        solver="lbfgs",
        max_iter=1000,
        random_state=RANDOM_STATE
    ),

    "decision_tree": DecisionTreeClassifier(
        max_depth=5,
        min_samples_leaf=100,
        random_state=RANDOM_STATE
    ),

    "random_forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=10,
        min_samples_leaf=100,
        max_features="sqrt",
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}

if XGBOOST_AVAILABLE:
    model_registry["xgboost"] = XGBClassifier(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

model_registry.keys()

dict_keys(['logistic_regression', 'decision_tree', 'random_forest'])

In [ ]:
# define candidate model-feature combinations
candidate_specs = [
    {
        "model_name": "logistic_regression",
        "feature_set": "A_core",
        "notes": "interpretable benchmark"
    },
    {
        "model_name": "logistic_regression",
        "feature_set": "B_interactions",
        "notes": "tests explicit interactions"
    },
    {
        "model_name": "logistic_regression",
        "feature_set": "C_expanded",
        "notes": "expanded logistic model"
    },
    {
        "model_name": "decision_tree",
        "feature_set": "A_core",
        "notes": "simple nonlinear interpretable model"
    },
    {
        "model_name": "decision_tree",
        "feature_set": "C_expanded",
        "notes": "simple nonlinear expanded model"
    },
    {
        "model_name": "random_forest",
        "feature_set": "A_core",
        "notes": "stable nonlinear ensemble"
    },
    {
        "model_name": "random_forest",
        "feature_set": "C_expanded",
        "notes": "stable nonlinear expanded ensemble"
    }
]

if XGBOOST_AVAILABLE:
    candidate_specs.extend([
        {
            "model_name": "xgboost",
            "feature_set": "A_core",
            "notes": "boosted nonlinear model"
        },
        {
            "model_name": "xgboost",
            "feature_set": "C_expanded",
            "notes": "boosted nonlinear expanded model"
        }
    ])

candidate_specs_df = pd.DataFrame(candidate_specs)
display(candidate_specs_df)

,model_name,feature_set,notes
0,logistic_regression,A_core,interpretable benchmark
1,logistic_regression,B_interactions,tests explicit interactions
2,logistic_regression,C_expanded,expanded logistic model
3,decision_tree,A_core,simple nonlinear interpretable model
4,decision_tree,C_expanded,simple nonlinear expanded model
5,random_forest,A_core,stable nonlinear ensemble
6,random_forest,C_expanded,stable nonlinear expanded ensemble


In [55]:
# validate candidate model-feature combinations
for spec in candidate_specs:
    model_name = spec["model_name"]
    feature_set_name = spec["feature_set"]

    assert model_name in model_registry, f"Model not found in registry: {model_name}"
    assert feature_set_name in feature_sets, f"Feature set not found: {feature_set_name}"
    assert feature_set_name in preprocessors, f"Preprocessor not found: {feature_set_name}"

print("Candidate specs validation passed.")

Candidate specs validation passed.


## 4.9 Build preprocessing + model pipelines

In [56]:
# build model pipeline factory
def build_model_pipeline(model, feature_set_name):
    """
    Build a sklearn pipeline containing:
    preprocessing recipe for selected feature set + model.
    
    Important:
    The preprocessing step is not fitted here.
    It will be fitted only inside each CV training fold.
    """
    pipeline = Pipeline(steps=[
        ("preprocessing", preprocessors[feature_set_name]),
        ("model", model)
    ])
    
    return pipeline

In [57]:
# build candidate pipeline objects
candidate_pipelines = {}

for spec in candidate_specs:
    model_name = spec["model_name"]
    feature_set_name = spec["feature_set"]
    
    candidate_key = f"{model_name}__{feature_set_name}"
    
    candidate_pipelines[candidate_key] = build_model_pipeline(
        model=model_registry[model_name],
        feature_set_name=feature_set_name
    )

print("Number of candidate pipelines:", len(candidate_pipelines))

for key in candidate_pipelines.keys():
    print(key)

Number of candidate pipelines: 7
logistic_regression__A_core
logistic_regression__B_interactions
logistic_regression__C_expanded
decision_tree__A_core
decision_tree__C_expanded
random_forest__A_core
random_forest__C_expanded


In [58]:
# validate candidate pipelines
for key, pipeline in candidate_pipelines.items():
    assert isinstance(pipeline, Pipeline), f"{key} is not a sklearn Pipeline"
    assert "preprocessing" in pipeline.named_steps, f"{key} missing preprocessing step"
    assert "model" in pipeline.named_steps, f"{key} missing model step"

print("Candidate pipeline validation passed.")

Candidate pipeline validation passed.


## 4.10 Smoke test

### 4.10.1 Logistic regression + A_core smoke test

In [ ]:
smoke_key = "logistic_regression__A_core"
smoke_model = candidate_pipelines[smoke_key]

X_smoke = trainval_df[feature_sets["A_core"]]
y_smoke = trainval_df[TARGET_COL]
groups_smoke = trainval_df[GROUP_COL]

first_train_idx, first_val_idx = next(
    cv_strategy.split(X_smoke, y_smoke, groups_smoke)
)

X_train_smoke = X_smoke.iloc[first_train_idx]
X_val_smoke = X_smoke.iloc[first_val_idx]

y_train_smoke = y_smoke.iloc[first_train_idx]
y_val_smoke = y_smoke.iloc[first_val_idx]

start_time = time.time()

smoke_model.fit(X_train_smoke, y_train_smoke)

y_pred_smoke = smoke_model.predict(X_val_smoke)
y_proba_smoke = smoke_model.predict_proba(X_val_smoke)[:, 1]

smoke_metrics = calculate_binary_metrics(
    y_true=y_val_smoke,
    y_pred=y_pred_smoke,
    y_proba=y_proba_smoke
)

runtime_seconds = time.time() - start_time

print("Smoke test model:", smoke_key)
print("Smoke test metrics:")
print(smoke_metrics)
print("Runtime seconds:", round(runtime_seconds, 2))

Smoke test model: logistic_regression__A_core
Smoke test metrics:
{'ROC_AUC': 0.8349520109743288, 'PR_AUC': 0.8737134489777956, 'accuracy': 0.7726315789473684, 'balanced_accuracy': 0.750943647645651, 'brier': 0.15637590261741272, 'log_loss': 0.4810260040764564}
Runtime seconds: 5.19


### 4.10.2 Decision tree + A_core smoke tesst

In [ ]:
smoke_tree_key = "decision_tree__A_core"
smoke_tree_model = candidate_pipelines[smoke_tree_key]

start_time = time.time()

smoke_tree_model.fit(X_train_smoke, y_train_smoke)

y_pred_tree_smoke = smoke_tree_model.predict(X_val_smoke)
y_proba_tree_smoke = smoke_tree_model.predict_proba(X_val_smoke)[:, 1]

smoke_tree_metrics = calculate_binary_metrics(
    y_true=y_val_smoke,
    y_pred=y_pred_tree_smoke,
    y_proba=y_proba_tree_smoke
)

runtime_seconds_tree = time.time() - start_time

print("Smoke test model:", smoke_tree_key)
print("Decision tree smoke test metrics:")
print(smoke_tree_metrics)
print("Runtime seconds:", round(runtime_seconds_tree, 2))

Smoke test model: decision_tree__A_core
Decision tree smoke test metrics:
{'ROC_AUC': 0.8304907562213554, 'PR_AUC': 0.857637495719286, 'accuracy': 0.7696842105263157, 'balanced_accuracy': 0.7535165830625092, 'brier': 0.15689415030545126, 'log_loss': 0.4816826743825245}
Runtime seconds: 3.59


## 4.11 Run broad CV screening

In [ ]:
# evaluate one candidate model with CV
def evaluate_candidate_cv(spec):
    model_name = spec["model_name"]
    feature_set_name = spec["feature_set"]
    notes = spec["notes"]
    
    candidate_key = f"{model_name}__{feature_set_name}"
    pipeline = candidate_pipelines[candidate_key]
    
    X = trainval_df[feature_sets[feature_set_name]]
    y = trainval_df[TARGET_COL]
    groups = trainval_df[GROUP_COL]
    
    start_time = time.time()
    
    fold_results = evaluate_cv_model(
        model=pipeline,
        X=X,
        y=y,
        groups=groups,
        cv_strategy=cv_strategy,
        metric_names=metric_names
    )
    
    runtime_seconds = time.time() - start_time
    
    summary = {
        "target": TARGET_COL,
        "model_name": model_name,
        "feature_set": feature_set_name,
        "candidate_key": candidate_key,
        "notes": notes,
        "runtime_seconds": runtime_seconds
    }
    
    for metric in metric_names:
        summary[f"mean_{metric}"] = fold_results[metric].mean()
        summary[f"std_{metric}"] = fold_results[metric].std()
    
    return summary, fold_results

In [ ]:
# run broad CV screening
screening_summaries = []
screening_fold_results = {}

for i, spec in enumerate(candidate_specs, start=1):
    candidate_key = f"{spec['model_name']}__{spec['feature_set']}"
    
    print("=" * 80)
    print(f"Running candidate {i}/{len(candidate_specs)}: {candidate_key}")
    
    try:
        summary, fold_results = evaluate_candidate_cv(spec)
        screening_summaries.append(summary)
        screening_fold_results[candidate_key] = fold_results
        
        print("Done.")
        print("Mean ROC-AUC:", round(summary["mean_ROC_AUC"], 4))
        print("Mean PR-AUC:", round(summary["mean_PR_AUC"], 4))
        print("Runtime seconds:", round(summary["runtime_seconds"], 2))
        
    except Exception as e:
        print("FAILED:", candidate_key)
        print("Error:", e)

Running candidate 1/7: logistic_regression__A_core
Done.
Mean ROC-AUC: 0.8142
Mean PR-AUC: 0.821
Runtime seconds: 20.39
Running candidate 2/7: logistic_regression__B_interactions
Done.
Mean ROC-AUC: 0.8159
Mean PR-AUC: 0.8256
Runtime seconds: 16.02
Running candidate 3/7: logistic_regression__C_expanded
Done.
Mean ROC-AUC: 0.8219
Mean PR-AUC: 0.8369
Runtime seconds: 38.1
Running candidate 4/7: decision_tree__A_core
Done.
Mean ROC-AUC: 0.8195
Mean PR-AUC: 0.816
Runtime seconds: 12.69
Running candidate 5/7: decision_tree__C_expanded
Done.
Mean ROC-AUC: 0.8262
Mean PR-AUC: 0.832
Runtime seconds: 17.45
Running candidate 6/7: random_forest__A_core
Done.
Mean ROC-AUC: 0.8302
Mean PR-AUC: 0.845
Runtime seconds: 61.1
Running candidate 7/7: random_forest__C_expanded
Done.
Mean ROC-AUC: 0.8364
Mean PR-AUC: 0.8592
Runtime seconds: 73.29


In [63]:
# create screening results table
screening_results_df = pd.DataFrame(screening_summaries)

screening_results_df = screening_results_df.sort_values(
    by="mean_ROC_AUC",
    ascending=False
).reset_index(drop=True)

display(screening_results_df)

,target,model_name,feature_set,candidate_key,notes,runtime_seconds,mean_ROC_AUC,std_ROC_AUC,mean_PR_AUC,std_PR_AUC,mean_accuracy,std_accuracy,mean_balanced_accuracy,std_balanced_accuracy,mean_brier,std_brier,mean_log_loss,std_log_loss
0,open,random_forest,C_expanded,random_forest__C_expanded,stable nonlinear expanded ensemble,73.294483,0.836392,0.026861,0.859215,0.037682,0.763761,0.028511,0.753004,0.031621,0.163052,0.017376,0.496900,0.043166
1,open,random_forest,A_core,random_forest__A_core,stable nonlinear ensemble,61.100862,0.830207,0.027462,0.844957,0.042158,0.764031,0.028647,0.751236,0.030757,0.165146,0.016516,0.504229,0.039897
2,open,decision_tree,C_expanded,decision_tree__C_expanded,simple nonlinear expanded model,17.451925,0.826182,0.036542,0.832031,0.054667,0.761653,0.026444,0.752640,0.030120,0.166246,0.019685,0.513710,0.053633
3,open,logistic_regression,C_expanded,logistic_regression__C_expanded,expanded logistic model,38.098379,0.821929,0.026887,0.836937,0.051186,0.753391,0.025303,0.742066,0.029898,0.170663,0.017311,0.519516,0.046379
4,open,decision_tree,A_core,decision_tree__A_core,simple nonlinear interpretable model,12.689436,0.819507,0.034983,0.816039,0.055258,0.760199,0.023182,0.750755,0.026124,0.166817,0.017365,0.517497,0.049015
5,open,logistic_regression,B_interactions,logistic_regression__B_interactions,tests explicit interactions,16.015746,0.815858,0.029914,0.825610,0.055860,0.751534,0.026536,0.739814,0.031163,0.172656,0.018606,0.526053,0.049862
6,open,logistic_regression,A_core,logistic_regression__A_core,interpretable benchmark,20.392323,0.814210,0.031477,0.820981,0.062358,0.752341,0.024453,0.740813,0.028422,0.172750,0.018524,0.526848,0.050479


In [64]:
# compact screening leaderboard
leaderboard_cols = [
    "model_name",
    "feature_set",
    "mean_ROC_AUC",
    "std_ROC_AUC",
    "mean_PR_AUC",
    "std_PR_AUC",
    "mean_accuracy",
    "mean_balanced_accuracy",
    "mean_brier",
    "mean_log_loss",
    "runtime_seconds",
    "notes"
]

display(screening_results_df[leaderboard_cols])

,model_name,feature_set,mean_ROC_AUC,std_ROC_AUC,mean_PR_AUC,std_PR_AUC,mean_accuracy,mean_balanced_accuracy,mean_brier,mean_log_loss,runtime_seconds,notes
0,random_forest,C_expanded,0.836392,0.026861,0.859215,0.037682,0.763761,0.753004,0.163052,0.496900,73.294483,stable nonlinear expanded ensemble
1,random_forest,A_core,0.830207,0.027462,0.844957,0.042158,0.764031,0.751236,0.165146,0.504229,61.100862,stable nonlinear ensemble
2,decision_tree,C_expanded,0.826182,0.036542,0.832031,0.054667,0.761653,0.752640,0.166246,0.513710,17.451925,simple nonlinear expanded model
3,logistic_regression,C_expanded,0.821929,0.026887,0.836937,0.051186,0.753391,0.742066,0.170663,0.519516,38.098379,expanded logistic model
4,decision_tree,A_core,0.819507,0.034983,0.816039,0.055258,0.760199,0.750755,0.166817,0.517497,12.689436,simple nonlinear interpretable model
5,logistic_regression,B_interactions,0.815858,0.029914,0.825610,0.055860,0.751534,0.739814,0.172656,0.526053,16.015746,tests explicit interactions
6,logistic_regression,A_core,0.814210,0.031477,0.820981,0.062358,0.752341,0.740813,0.172750,0.526848,20.392323,interpretable benchmark


## 4.12 Save Stage 4 outputs

In [1]:
# define Stage 4 output paths
BASELINE_RESULTS_PATH = OUTPUT_DIR / "4_baseline_results_open_v1.csv"
SCREENING_RESULTS_PATH = OUTPUT_DIR / "4_screening_results_open_v1.csv"
CV_FOLD_SUMMARY_PATH = OUTPUT_DIR / "4_cv_fold_summary_open_v1.csv"
CANDIDATE_SPECS_PATH = OUTPUT_DIR / "4_candidate_specs_open_v1.csv"
STAGE4_NOTES_PATH = OUTPUT_DIR / "4_modelling_stage4_notes_open_v1.md"

print(BASELINE_RESULTS_PATH)
print(SCREENING_RESULTS_PATH)
print(CV_FOLD_SUMMARY_PATH)
print(CANDIDATE_SPECS_PATH)
print(STAGE4_NOTES_PATH)

NameError: name 'OUTPUT_DIR' is not defined

In [ ]:
# save Stage 4 CSV outputs
baseline_results_df.to_csv(BASELINE_RESULTS_PATH, index=False)
screening_results_df.to_csv(SCREENING_RESULTS_PATH, index=False)
cv_fold_summary_open.to_csv(CV_FOLD_SUMMARY_PATH, index=False)
candidate_specs_df.to_csv(CANDIDATE_SPECS_PATH, index=False)

print("Saved baseline results:", BASELINE_RESULTS_PATH.exists())
print("Saved screening results:", SCREENING_RESULTS_PATH.exists())
print("Saved CV fold summary:", CV_FOLD_SUMMARY_PATH.exists())
print("Saved candidate specs:", CANDIDATE_SPECS_PATH.exists())

Saved baseline results: True
Saved screening results: True
Saved CV fold summary: True
Saved candidate specs: True


In [ ]:
# save Stage 4 notes
stage4_notes = f"""
# Modelling Stage 4 Notes — Open Prediction Screening v1

## Purpose

Stage 4 is the first actual modelling stage for open prediction. The purpose is to turn the frozen open dataset and preprocessing design into working sklearn model pipelines, run dummy baselines, run broad candidate screening, and produce the first cross-validation comparison table.

This stage does not perform deep hyperparameter tuning, final test evaluation, SHAP interpretation, or final model selection.

## Dataset

Input dataset:

- df_open_model_v1.parquet

Target:

- {TARGET_COL}

Group column:

- {GROUP_COL}

Unit of analysis:

- one user-campaign observation

## Split design

The data was split by campaign using mailing_id as the chronological proxy.

- Earliest 80% campaigns: train-validation pool
- Latest 20% campaigns: final untouched test set

Final test was created but not used for model selection or performance reporting in Stage 4.

## Train-validation / test summary

{split_summary_open.to_markdown(index=False)}

## Cross-validation

Inner CV strategy:

- StratifiedGroupKFold
- n_splits = {INNER_CV_SPLITS}
- shuffle = True
- random_state = {RANDOM_STATE}
- groups = mailing_id

Fold quality checks passed:

- no campaign overlap between train and validation fold
- each validation fold contains both open and non-open observations

## Feature sets

Feature sets used:

- A_core: clean thesis-aligned interpretable feature set
- B_interactions: A_core plus two explicit interaction features
- C_expanded: broader predictive feature set with raw counts/rates and additional content indicators

Feature counts:

- A_core: {len(feature_set_A_core)}
- B_interactions: {len(feature_set_B_interactions)}
- C_expanded: {len(feature_set_C_expanded)}

Interaction features in B:

- campaign_topic_x_age_group
- historical_open_bucket_x_prior_email_frequency_bucket

## Baselines

Two dummy baselines were evaluated:

- dummy_majority: predicts the most frequent class
- dummy_prior: predicts the training class prior probability

Baseline results:

{baseline_results_df.to_markdown(index=False)}

## Candidate models screened

Candidate models:

{candidate_specs_df.to_markdown(index=False)}

All real models were wrapped as sklearn pipelines:

preprocessing + model

This ensures that imputation, scaling, and one-hot encoding are fitted only inside each CV training fold.

## Screening results

Main screening leaderboard, sorted by mean ROC-AUC:

{screening_results_df[leaderboard_cols].to_markdown(index=False)}

## Main early observations

1. All real models clearly outperform the dummy baselines.
2. Logistic regression with A_core already performs strongly, suggesting the clean thesis-aligned features contain substantial predictive signal.
3. Adding explicit interactions improves logistic regression only slightly.
4. C_expanded improves performance more than B_interactions, suggesting raw/count/rate behavioural variables contain useful predictive information.
5. Decision tree models perform similarly to logistic regression but do not strongly dominate it.
6. Random forest models achieve the strongest screening performance, especially with C_expanded.
7. Differences among top candidates are modest relative to cross-validation variability, so final claims should wait until later tuning and final test evaluation.

## XGBoost status

XGBoost was not available in the current Python environment during Stage 4. It was not included in this screening run.

XGBoost may be added later as an optional boosted-tree extension after the current screening outputs are saved and reviewed.

## Temporal feature status

Temporal variables were not included in A/B/C feature sets during Stage 4.

If temporal features are added later, they should be introduced as a documented new feature set, for example D_temporal, rather than silently modifying A/B/C.

## Next step

Proceed to Stage 5:

- shortlist candidates for tuning
- likely keep logistic regression C_expanded as the main interpretable benchmark
- likely keep random forest A_core/C_expanded as strongest predictive candidates
- consider whether XGBoost should be installed and added as an optional comparison
- do not use the final test set until final model evaluation
"""

with open(STAGE4_NOTES_PATH, "w", encoding="utf-8") as f:
    f.write(stage4_notes)

print("Saved Stage 4 notes:", STAGE4_NOTES_PATH.exists())

Saved Stage 4 notes: True


In [ ]:
# final save check

saved_files = [
    BASELINE_RESULTS_PATH,
    SCREENING_RESULTS_PATH,
    CV_FOLD_SUMMARY_PATH,
    CANDIDATE_SPECS_PATH,
    STAGE4_NOTES_PATH
]

for path in saved_files:
    print(path.name, "exists:", path.exists())

baseline_results_open_v1.csv exists: True
screening_results_open_v1.csv exists: True
cv_fold_summary_open_v1.csv exists: True
candidate_specs_open_v1.csv exists: True
modelling_stage4_notes_open_v1.md exists: True
